# 01e · SMLP4Rec plain (history only, no query token), same training set as 01d, early stopping

Reference model for the final model (01d: query token + sameDest). The architecture is the history-only SMLP4Rec of notebook 01; what changes against nb01 is the **training set and the stopping rule**, so that the comparison with 01d isolates the query token:

- **Same training set as 01d**: the 1,710,952 warm next-booking targets of the train split **plus** the 713,754 first bookings (L = 0) inside the train window. A first booking has an empty history, so the plain model sees an all-padding sequence and can only learn the first-booking distribution. Counts are asserted against the 01d result file.
- **Same stopping rule as 01d**: up to `max_epochs` (10), early stopping on **valid warm NDCG@10** with `early_stop_patience` and `early_stop_min_delta` from `configs/smlprec_expedia.yaml` (`nbp.models.early_stop`). Best epoch = best valid epoch; valid NLL and the train/valid gap are logged as overfitting signals.
- **Same config, seed, split, rows and scoring**: 2 layers, hidden 64, dropout 0, CE, Adam lr 1e-3, batch 1024, seed 2022; warm test rows (L >= 1) and cold test rows (L = 0, first bookings), full ranking over 100 clusters, Recall@K / NDCG@K for K = 5, 10, 20.
- **Not used**: any query field, `destinations.csv`, hotel fields (`hotel_*`), `orig_destination_distance`.

Limitation: `hotel_cluster` is a proxy for room category / package; one dataset; offline only. Kernel: `smlp4rec`. Output: `results/week4_rebuild/smlprec_plain_cold_train_es_run.json`, run folder in `experiments/`.

In [1]:
import functools
import json
import os
import logging
import subprocess
import sys
import time
from datetime import date
from pathlib import Path

import numpy as np
import torch  # import torch before pandas: on Windows, pandas first makes c10.dll fail to load

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT), str(ROOT / "src")]
torch.load = functools.partial(torch.load, weights_only=False)  # RecBole checkpoints store the Config

import pandas as pd
from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.utils import init_seed

from nbp.eval.metrics import ndcg_at_k, summarize, target_rank
from nbp.models.early_stop import EarlyStopper, mean_nll
from src.models.smlprec import SMLPREC

torch.autograd.set_detect_anomaly(False)
logging.disable(logging.INFO)

CONFIG = ROOT / "configs" / "smlprec_expedia.yaml"
WORK = ROOT / "data" / "interim" / "recbole"
RES = ROOT / "results" / "week4_rebuild"
PLAIN_JSON = RES / "smlprec_plain_run.json"  # nb01 on the collapsed data (split sizes to match)
FINAL_JSON = RES / "smlprec_query_cold_train_es_run.json"  # 01d with early stopping (train-set sizes to match)
FINAL_COLD_JSON = RES / "smlprec_query_cold_L0_cold_train_es.json"  # cold row counts of the final model
SEED = int(os.environ.get("NBP_SEED", "2022"))
SUFFIX = "" if SEED == 2022 else f"-s{SEED}"
RESULT_JSON = RES / f"smlprec_plain_cold_train_es_run{SUFFIX.replace('-s', '_seed')}.json"
TAG = "plain-1e-clean-cold-es" + SUFFIX
RUN_DIR = ROOT / "experiments" / f"{date.today():%Y-%m-%d}_expedia_smlp4rec_{TAG}"
RUN_DIR.mkdir(parents=True, exist_ok=True)
RESUME = False  # True: continue from RUN_DIR/last.pth (weights + optimizer + per-epoch rows)
KS = (5, 10, 20)
print("run dir:", RUN_DIR)

run dir: C:\workspace\PROJECTS\next-best-product-recommendation\experiments\2026-10-10_expedia_smlp4rec_plain-1e-clean-cold-es


## 1. Config and data (same RecBole config, split and seed as nb01 / 01d)

In [2]:
config = Config(
    model=SMLPREC,
    dataset="expedia_clean_rowid",
    config_file_list=[str(CONFIG)],
    config_dict={
        "data_path": str(WORK),
        "checkpoint_dir": str(RUN_DIR),
        "load_col": {"inter": ["user_id", "item_id", "timestamp", "row_id"]},
        "seed": SEED,
    },
)
config["epochs"] = config["max_epochs"]  # train up to max_epochs with early stopping (yaml `epochs` stays 3 for nb01)
init_seed(config["seed"], config["reproducibility"])

t0 = time.time()
dataset = create_dataset(config)
train_data, valid_data, test_data = data_preparation(config, dataset)
t_data = time.time() - t0

ITEM_SEQ = config["ITEM_ID_FIELD"] + config["LIST_SUFFIX"]
ITEM_SEQ_LEN = config["ITEM_LIST_LENGTH_FIELD"]
IID, TIME, UID = config["ITEM_ID_FIELD"], config["TIME_FIELD"], config["USER_ID_FIELD"]
S_LEN = config["MAX_ITEM_LIST_LENGTH"]


def split_inputs(loader) -> dict:
    # history ids (n, 20), history length (n,), true next cluster (n,)
    d = loader.dataset.inter_feat
    return {"seq": d[ITEM_SEQ], "length": d[ITEM_SEQ_LEN], "y": d[IID].numpy()}


splits = {}
for name, loader in (("train", train_data), ("valid", valid_data), ("test", test_data)):
    d = loader.dataset.inter_feat
    splits[name] = {"targets": len(d[TIME]), "first_target_event": int(d[TIME].min()),
                    "last_target_event": int(d[TIME].max()),
                    "mean_history_len": round(float(d[ITEM_SEQ_LEN].float().mean()), 3)}
print(f"data build {t_data:.1f}s; users {dataset.user_num - 1}, clusters {dataset.item_num - 1}")
print(json.dumps(splits, indent=1))

plain_ref = json.loads(PLAIN_JSON.read_text(encoding="utf-8"))
final_ref = json.loads(FINAL_JSON.read_text(encoding="utf-8"))
for name in ("train", "valid", "test"):
    for k in ("targets", "first_target_event", "last_target_event", "mean_history_len"):
        assert splits[name][k] == plain_ref["splits"][name][k] == final_ref["splits"][name][k], (name, k)
print("warm split identical to nb01 and to the final model (01d)")

data build 50.2s; users 813985, clusters 100
{
 "train": {
  "targets": 1710952,
  "first_target_event": 20,
  "last_target_event": 2424705,
  "mean_history_len": 5.781
 },
 "valid": {
  "targets": 213869,
  "first_target_event": 2424706,
  "last_target_event": 2689343,
  "mean_history_len": 7.305
 },
 "test": {
  "targets": 213869,
  "first_target_event": 2689344,
  "last_target_event": 2952674,
  "mean_history_len": 7.166
 }
}
warm split identical to nb01 and to the final model (01d)


## 2. Cold rows (L = 0) and the training set

Built exactly as in 01d / `scripts/query_token_cold_users.py`: the first booking of every user. Those inside the train window (event index below the last train target) are **training rows with an empty history**; those whose first booking falls in the valid / test window are the cold evaluation rows. An empty history is passed as one all-padding position (`length = 1`) because RecBole's `gather_indexes` reads position `length - 1`.

In [3]:
tok2id = dataset.field2token_id[dataset.iid_field]
inter = pd.read_csv(
    WORK / "expedia_clean_rowid" / "expedia_clean_rowid.inter", sep="\t",
    usecols=["user_id:token", "item_id:token", "timestamp:float", "row_id:float"],
).sort_values(["user_id:token", "timestamp:float"], kind="stable")
first = inter.drop_duplicates("user_id:token", keep="first")
ts_first = first["timestamp:float"]
t_train_last, t_valid_last = splits["train"]["last_target_event"], splits["valid"]["last_target_event"]


def cold_part(df: pd.DataFrame) -> dict:
    y = np.array([tok2id[str(int(k))] for k in df["item_id:token"]])
    return {"seq": torch.zeros((len(df), S_LEN), dtype=torch.long), "length": torch.ones(len(df), dtype=torch.long), "y": y}


cold = {
    "train": cold_part(first[ts_first < t_train_last]),
    "valid": cold_part(first[(ts_first >= t_train_last) & (ts_first < t_valid_last)]),
    "test": cold_part(first[ts_first >= t_valid_last]),
}
n_cold = {k: len(v["y"]) for k, v in cold.items()}
cold_ref = json.loads(FINAL_COLD_JSON.read_text(encoding="utf-8"))["n_rows"]
assert n_cold["train"] == final_ref["cold_training_rows"], n_cold
assert n_cold["valid"] == cold_ref["valid"] and n_cold["test"] == cold_ref["test"], (n_cold, cold_ref)
print("cold rows:", n_cold, "- identical to the final model")

d_tr = train_data.dataset.inter_feat
TRAIN = {  # warm targets first, then the empty-history rows (the 01d training set)
    "seq": torch.cat([d_tr[ITEM_SEQ], cold["train"]["seq"]]),
    "length": torch.cat([d_tr[ITEM_SEQ_LEN], cold["train"]["length"]]),
    "y": torch.cat([d_tr[IID], torch.from_numpy(cold["train"]["y"]).to(d_tr[IID].dtype)]),
}
assert len(TRAIN["y"]) == splits["train"]["targets"] + final_ref["cold_training_rows"]
TRAIN_GEN = torch.Generator().manual_seed(config["seed"])
EVAL = {  # scored after every epoch; only valid_warm drives early stopping
    "valid_warm": split_inputs(valid_data), "valid_cold": cold["valid"],
    "test_warm": split_inputs(test_data), "test_cold": cold["test"],
}
print({k: tuple(v.shape) for k, v in TRAIN.items()}, {k: len(v["y"]) for k, v in EVAL.items()})

cold rows: {'train': 713754, 'valid': 50769, 'test': 49462} - identical to the final model
{'seq': (2424706, 20), 'length': (2424706,), 'y': (2424706,)} {'valid_warm': 213869, 'valid_cold': 50769, 'test_warm': 213869, 'test_cold': 49462}


## 3. Model, training and scoring functions

In [4]:
model = SMLPREC(config, train_data.dataset).to(config["device"])
optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"], weight_decay=config["weight_decay"])
n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"parameters: {n_params} (history-only model nb01: {plain_ref['n_parameters']})")
assert n_params == plain_ref["n_parameters"]


def train_epoch(model, optimizer) -> float:
    # one epoch over warm targets + empty-history rows in random order; returns the sum of batch losses
    model.train()
    total, n = 0.0, len(TRAIN["y"])
    perm = torch.randperm(n, generator=TRAIN_GEN)
    for i in range(0, n, config["train_batch_size"]):
        idx = perm[i : i + config["train_batch_size"]]
        inter = {ITEM_SEQ: TRAIN["seq"][idx], ITEM_SEQ_LEN: TRAIN["length"][idx], IID: TRAIN["y"][idx]}
        optimizer.zero_grad()
        loss = model.calculate_loss(inter)
        if torch.isnan(loss):
            raise ValueError("training loss is nan")
        loss.backward()
        optimizer.step()
        total += loss.item()
    return total


@torch.no_grad()
def full_scores(model, inp: dict, seq=None, batch: int = 8192) -> np.ndarray:
    # (n, n_items) scores; padding column = -inf. `seq` overrides the history (control)
    model.eval()
    seq = inp["seq"] if seq is None else seq
    out = []
    for i in range(0, len(inp["y"]), batch):
        s = model.full_sort_predict({ITEM_SEQ: seq[i : i + batch], ITEM_SEQ_LEN: inp["length"][i : i + batch]})
        s[:, 0] = float("-inf")
        out.append(s.cpu().numpy())
    return np.concatenate(out)

SMLPREC(
  (item_embedding): Embedding(101, 64, padding_idx=0)
  (layers): ModuleList(
    (0): SMLP(
      (c): Sequential(
        (0): Linear(in_features=1, out_features=3, bias=False)
        (1): GELU(approximate='none')
        (2): Dropout(p=0.0, inplace=False)
        (3): Linear(in_features=3, out_features=1, bias=False)
        (4): Dropout(p=0.0, inplace=False)
      )
      (h): Sequential(
        (0): Linear(in_features=20, out_features=60, bias=False)
        (1): GELU(approximate='none')
        (2): Dropout(p=0.0, inplace=False)
        (3): Linear(in_features=60, out_features=20, bias=False)
        (4): Dropout(p=0.0, inplace=False)
      )
      (w): Sequential(
        (0): Linear(in_features=64, out_features=192, bias=False)
        (1): GELU(approximate='none')
        (2): Dropout(p=0.0, inplace=False)
        (3): Linear(in_features=192, out_features=64, bias=False)
        (4): Dropout(p=0.0, inplace=False)
      )
      (norm2): LayerNorm((64,), eps=1e-05, el

## 4. Train (up to `max_epochs`, early stopping)

Same rule as 01d: the monitor is **valid warm NDCG@10 only** (valid cold and test are recorded, never used to decide); an epoch is an improvement only if it beats the best by more than `early_stop_min_delta`; stop after `early_stop_patience` epochs without one. Valid NLL is logged: if train loss keeps falling while valid NLL rises, the model is over-fitting even when ranking metrics hold.

In [5]:
per_epoch, start = [], 1
if RESUME and (RUN_DIR / "last.pth").exists():
    ck = torch.load(RUN_DIR / "last.pth")
    model.load_state_dict(ck["state_dict"])
    optimizer.load_state_dict(ck["optimizer"])
    per_epoch, start = ck["per_epoch"], ck["epoch"] + 1
    print("resumed after epoch", ck["epoch"])
stopper = EarlyStopper.from_scores(
    [r["monitor"] for r in per_epoch], config["early_stop_patience"], config["early_stop_min_delta"]
)
n_batches = int(np.ceil(len(TRAIN["y"]) / config["train_batch_size"]))

for epoch in range(start, config["epochs"] + 1):
    if stopper.should_stop:
        break
    t = time.time()
    loss = train_epoch(model, optimizer)
    row = {"epoch": epoch, "train_loss": round(loss, 4), "train_loss_per_batch": round(loss / n_batches, 4),
           "train_seconds": round(time.time() - t, 1)}
    t = time.time()
    ranks = {}
    for name, inp in EVAL.items():
        scores = full_scores(model, inp)
        ranks[name] = target_rank(scores, inp["y"])
        row[name] = summarize(ranks[name], ks=KS)
        if name == "valid_warm":
            row["valid_nll"] = round(mean_nll(scores, inp["y"]), 4)
    row["eval_seconds"] = round(time.time() - t, 1)
    row["monitor"] = ndcg_at_k(ranks["valid_warm"], 10)
    torch.save(model.state_dict(), RUN_DIR / f"epoch{epoch}.pth")
    stopper.update(epoch, row["monitor"])
    row["is_best"] = stopper.best_epoch == epoch
    per_epoch.append(row)
    torch.save({"epoch": epoch, "state_dict": model.state_dict(), "optimizer": optimizer.state_dict(),
                "per_epoch": per_epoch}, RUN_DIR / "last.pth")
    print(
        f"epoch {epoch}: loss/batch {row['train_loss_per_batch']:.4f} (train {row['train_seconds']}s, eval {row['eval_seconds']}s) | "
        f"valid NLL {row['valid_nll']:.4f} warm NDCG@10 {row['valid_warm']['ndcg@10']:.4f} cold NDCG@5 {row['valid_cold']['ndcg@5']:.4f} | "
        f"best {stopper.best_epoch}, no gain for {stopper.bad_epochs}/{stopper.patience}"
    )
    nll = [r["valid_nll"] for r in per_epoch]
    if len(nll) >= 3 and nll[-1] > nll[-2] > nll[-3]:
        print("WARNING: valid NLL rose for 2 epochs in a row while train loss fell - over-fitting onset")

stop_reason = (f"early stop: no valid NDCG@10 gain > {stopper.min_delta:g} for {stopper.patience} epochs"
               if stopper.should_stop else f"reached max_epochs = {config['epochs']}")
best = {"epoch": stopper.best_epoch, "valid_score": stopper.best_score}
for r in per_epoch:
    r["is_best"] = r["epoch"] == best["epoch"]
t_fit = sum(r["train_seconds"] + r["eval_seconds"] for r in per_epoch)
print(f"{stop_reason}; trained {len(per_epoch)} epochs; best epoch by valid warm NDCG@10: {best['epoch']}")
if best["epoch"] == config["epochs"]:
    print("WARNING: best epoch is the last allowed one - the model may be under-trained; raise max_epochs")

epoch 1: loss/batch 4.0939 (train 475.9s, eval 30.0s) | valid NLL 3.8831 warm NDCG@10 0.2687 cold NDCG@5 0.0799 | best 1, no gain for 0/2


epoch 2: loss/batch 4.0730 (train 459.2s, eval 29.3s) | valid NLL 3.8774 warm NDCG@10 0.2696 cold NDCG@5 0.0795 | best 2, no gain for 0/2


epoch 3: loss/batch 4.0702 (train 451.8s, eval 28.9s) | valid NLL 3.8742 warm NDCG@10 0.2698 cold NDCG@5 0.0782 | best 2, no gain for 1/2


epoch 4: loss/batch 4.0686 (train 434.8s, eval 29.4s) | valid NLL 3.8741 warm NDCG@10 0.2700 cold NDCG@5 0.0799 | best 2, no gain for 2/2
early stop: no valid NDCG@10 gain > 0.0007 for 2 epochs; trained 4 epochs; best epoch by valid warm NDCG@10: 2


## 5. Best epoch: save `best.pth`, score valid and test once (warm, cold, all events)

In [6]:
state = torch.load(RUN_DIR / f"epoch{best['epoch']}.pth")
model.load_state_dict(state)
ARCH_KEYS = ("MAX_ITEM_LIST_LENGTH", "n_layers", "hidden_size", "hidden_dropout_prob", "hidden_act",
             "layer_norm_eps", "initializer_range", "loss_type", "seed")
torch.save({"epoch": best["epoch"], "state_dict": state, "config": {k: config[k] for k in ARCH_KEYS}}, RUN_DIR / "best.pth")

rk = {name: target_rank(full_scores(model, inp), inp["y"]) for name, inp in EVAL.items()}
final = {}
for sp in ("valid", "test"):
    w, c = rk[f"{sp}_warm"], rk[f"{sp}_cold"]
    final[sp] = {"warm": summarize(w, ks=KS), "cold (L=0)": summarize(c, ks=KS), "all events": summarize(np.concatenate([w, c]), ks=KS)}
np.savez_compressed(RUN_DIR / "ranks_test.npz", warm=rk["test_warm"], cold=rk["test_cold"])
COLS = ["n", "recall@5", "ndcg@5", "recall@10", "ndcg@10", "recall@20", "ndcg@20"]
for sp in ("test", "valid"):
    print(f"\n{sp.upper()} (best epoch {best['epoch']}, Expedia collapsed bookings, full ranking over 100 clusters)")
    print(pd.DataFrame(final[sp]).T[COLS].to_string())


TEST (best epoch 2, Expedia collapsed bookings, full ranking over 100 clusters)
                   n  recall@5  ndcg@5  recall@10  ndcg@10  recall@20  ndcg@20
warm        213869.0    0.3214  0.2277     0.4532   0.2701     0.6230   0.3129
cold (L=0)   49462.0    0.1307  0.0809     0.2288   0.1124     0.3599   0.1454
all events  263331.0    0.2856  0.2001     0.4110   0.2405     0.5736   0.2814

VALID (best epoch 2, Expedia collapsed bookings, full ranking over 100 clusters)
                   n  recall@5  ndcg@5  recall@10  ndcg@10  recall@20  ndcg@20
warm        213869.0    0.3206  0.2276     0.4510   0.2696     0.6228   0.3128
cold (L=0)   50769.0    0.1238  0.0795     0.2116   0.1076     0.3595   0.1448
all events  264638.0    0.2828  0.1992     0.4051   0.2385     0.5723   0.2806


## 6. Verification (assert on failure)

In [7]:
checks = {}
# a. the plain model has no query input: an empty history must give the same scores for every cold row
sc = full_scores(model, EVAL["test_cold"])
checks["cold_scores_max_abs_spread_over_rows"] = float(np.abs(sc[:, 1:] - sc[0, 1:]).max())
assert checks["cold_scores_max_abs_spread_over_rows"] < 1e-4
# b. shuffled histories on warm test rows must hurt (the model really uses the history)
g = torch.Generator().manual_seed(0)
inp = EVAL["test_warm"]
perm = torch.randperm(len(inp["y"]), generator=g)
shuf_inp = {**inp, "length": inp["length"][perm]}
r_shuf = target_rank(full_scores(model, shuf_inp, seq=inp["seq"][perm]), inp["y"])
checks["test_warm_ndcg@5"] = {"real": final["test"]["warm"]["ndcg@5"], "histories shuffled": summarize(r_shuf, ks=(5,))["ndcg@5"]}
assert checks["test_warm_ndcg@5"]["histories shuffled"] < checks["test_warm_ndcg@5"]["real"]
# c. training set = the 01d training set; eval rows = the final model's eval rows
checks["train_rows"] = {"here": len(TRAIN["y"]), "warm": splits["train"]["targets"], "cold": n_cold["train"],
                        "final_model_cold_training_rows": final_ref["cold_training_rows"]}
checks["eval_rows_vs_final"] = {"test_warm": len(EVAL["test_warm"]["y"]) == final_ref["test"]["n"],
                               "test_cold": n_cold["test"] == cold_ref["test"]}
assert all(checks["eval_rows_vs_final"].values())
print(json.dumps(checks, indent=1))
print("all verification checks passed")

{
 "cold_scores_max_abs_spread_over_rows": 3.5762786865234375e-07,
 "test_warm_ndcg@5": {
  "real": 0.2277,
  "histories shuffled": 0.0542
 },
 "train_rows": {
  "here": 2424706,
  "warm": 1710952,
  "cold": 713754,
  "final_model_cold_training_rows": 713754
 },
 "eval_rows_vs_final": {
  "test_warm": true,
  "test_cold": true
 }
}
all verification checks passed


## 7. Save the result file

In [8]:
commit = subprocess.run(["git", "-C", str(ROOT), "rev-parse", "--short", "HEAD"], capture_output=True, text=True, check=False).stdout.strip()
result = {
    "run": "SMLP4Rec plain (history only, no query token), trained with warm + first-booking rows, up to max_epochs with early stopping, notebook 01e",
    "dataset": "expedia (bookings, burst repeats collapsed; item = hotel_cluster)",
    "data_version": "collapsed (expedia_bookings.parquet)",
    "git_commit": commit,
    "config": {k: config[k] for k in ("MAX_ITEM_LIST_LENGTH", "n_layers", "hidden_size", "hidden_dropout_prob", "loss_type", "epochs",
                                      "max_epochs", "early_stop_patience", "early_stop_min_delta", "train_batch_size",
                                      "learning_rate", "seed", "eval_args", "topk", "valid_metric")},
    "n_parameters": int(n_params),
    "splits": splits,
    "cold_rows": n_cold,
    "training_set": "warm train targets + first bookings inside the train window (empty history), identical to 01d",
    "timing_seconds": {"data_build": round(t_data, 1), "train_and_eval": round(t_fit, 1)},
    "best_epoch_by_valid": best["epoch"],
    "stop_reason": stop_reason,
    "early_stop": {"monitor": "valid warm ndcg@10", "patience": config["early_stop_patience"],
                   "min_delta": config["early_stop_min_delta"], "max_epochs": config["max_epochs"]},
    "checkpoint": str(RUN_DIR / "best.pth"),
    "per_epoch": per_epoch,
    "valid": final["valid"],
    "test": final["test"],
    "verification": checks,
}
RES.mkdir(parents=True, exist_ok=True)
for path in (RESULT_JSON, RUN_DIR / "metrics.json"):
    path.write_text(json.dumps(result, indent=2), encoding="utf-8")
print("wrote", RESULT_JSON, "and", RUN_DIR / "metrics.json")

wrote C:\workspace\PROJECTS\next-best-product-recommendation\results\week4_rebuild\smlprec_plain_cold_train_es_run.json and C:\workspace\PROJECTS\next-best-product-recommendation\experiments\2026-10-10_expedia_smlp4rec_plain-1e-clean-cold-es\metrics.json
